In [1]:
import numpy as np

class ConstantProductAMM:
    """
    Simulates a Constant Product Market Maker (Uniswap V2 logic).
    Formula: x * y = k
    """
    def __init__(self, reserve_x, reserve_y, fee=0.003):
        """
        Initialize the liquidity pool.
        
        Args:
            reserve_x (float): Quantity of Token X (e.g., ETH)
            reserve_y (float): Quantity of Token Y (e.g., USDC)
            fee (float): Trading fee (default 0.3% = 0.003)
        """
        self.reserve_x = float(reserve_x)
        self.reserve_y = float(reserve_y)
        self.fee = fee
        self.k = self.reserve_x * self.reserve_y
        self.initial_k = self.k # Store initial k for PL tracking
        
    def get_price_x_in_y(self):
        """Returns current price of Token X in terms of Token Y (Y per X)."""
        return self.reserve_y / self.reserve_x

    def get_amount_out(self, amount_in, reserve_in, reserve_out):
        """
        Calculates output amount given input amount using Uniswap V2 formula with fees.
        
        Formula:
        dy = (y * dx * 0.997) / (x + dx * 0.997)
        """
        amount_in_with_fee = amount_in * (1 - self.fee)
        numerator = amount_in_with_fee * reserve_out
        denominator = reserve_in + amount_in_with_fee
        return numerator / denominator

    def swap_x_for_y(self, delta_x):
        """
        Swap Token X (Input) for Token Y (Output).
        Ex: Sell ETH, receive USDC.
        """
        dy = self.get_amount_out(delta_x, self.reserve_x, self.reserve_y)
        
        # Update Reserves
        self.reserve_x += delta_x
        self.reserve_y -= dy
        self.k = self.reserve_x * self.reserve_y # k slightly increases due to fees
        
        return dy

    def swap_y_for_x(self, delta_y):
        """
        Swap Token Y (Input) for Token X (Output).
        Ex: Sell USDC, receive ETH.
        """
        dx = self.get_amount_out(delta_y, self.reserve_y, self.reserve_x)
        
        # Update Reserves
        self.reserve_y += delta_y
        self.reserve_x -= dx
        self.k = self.reserve_x * self.reserve_y
        
        return dx

    def calculate_price_impact(self, amount_in, is_x_to_y=True):
        """
        Calculates Price Impact (Slippage) for a potential trade.
        """
        if is_x_to_y:
            current_price = self.reserve_y / self.reserve_x
            estimated_out = self.get_amount_out(amount_in, self.reserve_x, self.reserve_y)
            execution_price = estimated_out / amount_in
        else:
            current_price = self.reserve_x / self.reserve_y
            estimated_out = self.get_amount_out(amount_in, self.reserve_y, self.reserve_x)
            execution_price = estimated_out / amount_in
            
        return abs(execution_price - current_price) / current_price

# --- VERIFICATION SCRIPT ---
if __name__ == "__main__":
    # Test Scenario: 100 ETH and 200,000 USDC Pool (Price = 2000 USDC/ETH)
    amm = ConstantProductAMM(reserve_x=100, reserve_y=200000)
    
    print(f"Initial State: {amm.reserve_x} ETH | {amm.reserve_y} USDC")
    print(f"Initial Price: {amm.get_price_x_in_y()} USDC/ETH")
    
    # 1. Trader buys ETH with 5000 USDC
    usdc_in = 5000
    eth_out = amm.swap_y_for_x(usdc_in)
    
    print("\n--- Trade 1: Swap 5000 USDC for ETH ---")
    print(f"Input: {usdc_in} USDC")
    print(f"Output: {eth_out:.6f} ETH")
    print(f"Effective Price: {usdc_in/eth_out:.2f} USDC/ETH (Price Impact included)")
    
    # 2. Check Pool State
    print("\n--- New Pool State ---")
    print(f"Reserves: {amm.reserve_x:.6f} ETH | {amm.reserve_y:.6f} USDC")
    print(f"New Market Price: {amm.get_price_x_in_y():.2f} USDC/ETH")

Initial State: 100.0 ETH | 200000.0 USDC
Initial Price: 2000.0 USDC/ETH

--- Trade 1: Swap 5000 USDC for ETH ---
Input: 5000 USDC
Output: 2.431885 ETH
Effective Price: 2056.02 USDC/ETH (Price Impact included)

--- New Pool State ---
Reserves: 97.568115 ETH | 205000.000000 USDC
New Market Price: 2101.10 USDC/ETH
